# Module 2: Projection Performance — Making Vertica Shine

**Course 2: Vertica Advanced Features for DDC Analytics**

---

## Prerequisites
- Module 1 (Dimensional Model) must be completed
- All projections have been created during initialization

## Setup

Import the helper functions we'll use throughout this notebook:

In [ ]:
from ddc_helpers import run_query, explain_query

---

## The Problem: Slow Spatial Queries

**Scenario:** During a dengue outbreak, epidemiologists need to track case clusters across subdistricts in real-time. A naive spatial join on 5 million patient records takes **47 seconds**.

After adding Vertica projections, the same query takes **0.8 seconds**.

### Why This Matters for DDC
- A 47-second query → epidemiologists switch to Excel
- A 0.8-second query → they stay in the data warehouse
- **Sub-second queries enable interactive exploration during outbreaks**

### The Key Insight

Vertica has **NO R-tree spatial index** (unlike PostGIS). Instead, it achieves similar performance through:

1. **Columnar storage** — only reads the columns you query
2. **Projection ordering** — physically sorts data on disk
3. **Data locality** — geographically close records sit in adjacent blocks

**Analogy:** Think of a library where the same books are shelved in three ways:
- By author name (for "find all books by Pramoedya")
- By subject (for "find all books about epidemiology")
- By publication date (for "find recent books")

Each arrangement is a **projection**. The library has 3 copies of every book, but each copy is sorted to answer a different question fast.

**Spatial indexes** say "look HERE for nearby data."

**Projections** say "nearby data is ALREADY next to each other on disk."

Both achieve data locality. Vertica just does it at the storage layer.

---

## EXPLAIN Plan: Understanding How Vertica Executes the Query

Let's examine the query plan for a spatial query that finds all infections within a bounding box around Bangkok.

The bounding box coordinates represent central Bangkok:
- Longitude: 100.4 to 100.7
- Latitude: 13.6 to 13.9

**Note:** Projections were already created during initialization, so Vertica will automatically choose the best ones for this query.

In [ ]:
explain_query("""
SELECT
    loc.province_name_th,
    loc.district_name_th,
    loc.subdistrict_name_th,
    dis.disease_name_th,
    COUNT(*)                          AS case_count,
    AVG(fi.severity_score)            AS avg_severity,
    SUM(CASE WHEN fi.outcome = 'deceased' THEN 1 ELSE 0 END) AS deaths
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
WHERE loc.is_current = TRUE
  AND dd.year_num = 2023
  AND ST_Intersects(
        loc.subdistrict_geom,
        ST_GeomFromText('POLYGON((100.4 13.6, 100.7 13.6, 100.7 13.9, 100.4 13.9, 100.4 13.6))')
      )
GROUP BY loc.province_name_th, loc.district_name_th, loc.subdistrict_name_th, dis.disease_name_th
ORDER BY case_count DESC
""")

### What to Look For in the EXPLAIN Output

Notice:
- **Projection names** — Vertica automatically chose the best projections
- **Join types** — MERGE JOIN (fast) vs HASH JOIN (slower)
- **Scan types** — Sorted range scan vs full table scan
- **Row counts** — Rows filtered early vs rows filtered late

Because our projections were created during initialization, you should see:
- `dim_location_spatial` (sorted by latitude, longitude)
- `fact_infection_by_location` (sorted by location_sk, infection_date_id)
- MERGE JOIN operations (enabled by sorted data)

---

## Run the Actual Query

Now let's execute the query and see the results:

In [ ]:
run_query("""
SELECT
    loc.province_name_th,
    loc.district_name_th,
    loc.subdistrict_name_th,
    dis.disease_name_th,
    COUNT(*)                          AS case_count,
    AVG(fi.severity_score)            AS avg_severity,
    SUM(CASE WHEN fi.outcome = 'deceased' THEN 1 ELSE 0 END) AS deaths
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
WHERE loc.is_current = TRUE
  AND dd.year_num = 2023
  AND ST_Intersects(
        loc.subdistrict_geom,
        ST_GeomFromText('POLYGON((100.4 13.6, 100.7 13.6, 100.7 13.9, 100.4 13.9, 100.4 13.6))')
      )
GROUP BY loc.province_name_th, loc.district_name_th, loc.subdistrict_name_th, dis.disease_name_th
ORDER BY case_count DESC
LIMIT 20
""")

---

## What Are Projections?

Projections are Vertica's secret weapon. They are **physically sorted copies** of your table data stored on disk. You can have **MULTIPLE projections** of the same table, each sorted differently for different query patterns.

### The 5 Projections Created for DDC Training

During initialization, these projections were created:

#### 1. `dim_date_by_date_id`
- **Sort order:** `ORDER BY date_id`
- **Why:** Most queries filter by date range (e.g., `WHERE dd.year_num = 2023`)
- **Benefit:** Vertica can skip irrelevant date blocks entirely
- **Segmentation:** `HASH(date_id)` for parallel execution

#### 2. `dim_location_by_sk`
- **Sort order:** `ORDER BY location_sk`
- **Why:** Fact table joins on `location_sk`
- **Benefit:** Enables fast MERGE JOIN with fact table
- **Segmentation:** `HASH(location_sk)`

#### 3. `dim_location_spatial` ⭐ **KEY INSIGHT**
- **Sort order:** `ORDER BY latitude, longitude`
- **Why:** Spatial queries need geographically close records together
- **Benefit:** Bangkok rows stored together on disk → spatial filter only reads Bangkok blocks
- **Segmentation:** `HASH(province_code)` — all subdistricts of a province on same node

**This is how Vertica does fast spatial queries without an R-tree:**
- Subdistricts in Bangkok (lat ~13.7) are stored together on disk
- Subdistricts in Chiang Mai (lat ~18.8) are stored together on disk
- A bounding box query for Bangkok only reads the Bangkok disk blocks
- This is called **"data locality through sort order"**

#### 4. `fact_infection_by_date`
- **Sort order:** `ORDER BY infection_date_id, location_sk`
- **Why:** Most DDC queries are "show me cases over time"
- **Benefit:** Time-range filters skip irrelevant date blocks; within a date range, location-based filtering is also fast
- **Segmentation:** `HASH(infection_date_id)`

#### 5. `fact_infection_by_location`
- **Sort order:** `ORDER BY location_sk, infection_date_id`
- **Why:** Spatial queries filter by location FIRST, then by time
- **Benefit:** All Bangkok cases together on disk
- **Example query:** "Show all dengue cases in Bangkok in 2023"
- **Segmentation:** `HASH(location_sk)`

---

## Inspect Projections

Let's verify that our projections are up-to-date and available:

In [ ]:
run_query("""
SELECT
    anchor_table_name   AS table_name,
    projection_name,
    is_up_to_date,
    verified_fault_tolerance
FROM projections
WHERE projection_schema = 'ddc_training'
  AND anchor_table_name IN ('dim_date', 'dim_location', 'dim_disease', 'fact_infection')
ORDER BY anchor_table_name, projection_name
""")

### What to Check
- **is_up_to_date:** Should be `t` (true) — means the projection has all current data
- **verified_fault_tolerance:** Should be `t` — means data is replicated for safety
- You should see all 5 custom projections plus any auto-created `_super` projections

---

## The Key Insight: dim_location_spatial

The `dim_location_spatial` projection is the key to fast spatial queries in Vertica.

### How It Works

When we `ORDER BY latitude, longitude`:
- Bangkok rows (latitude ~13.7, longitude ~100.5) are stored together on disk
- Chiang Mai rows (latitude ~18.8, longitude ~99.0) are stored together on disk
- A spatial query for Bangkok only reads the Bangkok disk blocks

### Data Locality Example

Without spatial ordering:
```
Disk blocks: [Bangkok, Songkhla, Bangkok, Chiang Mai, Bangkok, ...]
Query for Bangkok: Must scan ALL blocks (slow)
```

With spatial ordering:
```
Disk blocks: [All Bangkok rows] [All Chiang Mai rows] [All Songkhla rows]
Query for Bangkok: Only scan Bangkok blocks (fast)
```

This is NOT as flexible as an R-tree spatial index, but for regional queries it is remarkably effective — and it costs nothing at query time.

---

## Performance Comparison

### Before vs After Optimized Projections

| Metric | BEFORE (auto projection) | AFTER (custom projections) |
|--------|--------------------------|---------------------------|
| **Projection used** | `*_super` (unsorted) | `dim_location_spatial` (sorted by lat/long) |
| **Join type** | HASH JOIN | MERGE JOIN |
| **Scan type** | Full table scan | Sorted range scan |
| **Rows filtered** | Late (after join) | Early (during scan) |
| **Query time (5M rows)** | ~47 seconds | ~0.8 seconds |
| **Speedup** | — | **58x faster** |

### Why It Is Faster

1. **Spatial filter:** Vertica picks `dim_location_spatial` projection (sorted by lat/long)
   - Bangkok rows are contiguous on disk
   - Minimal I/O for spatial filter

2. **Fact join:** Vertica picks `fact_infection_by_location` projection (sorted by location_sk)
   - All infections for Bangkok locations are contiguous
   - Fast MERGE JOIN (no hash table build)

3. **Columnar storage:** Only the 7 columns in SELECT are read (not all 10)

4. **Statistics:** Enable the optimizer to choose MERGE JOIN over HASH JOIN

---

## DDC Rule of Thumb for Projections

### When to Create Projections

- **Create a date-ordered projection** for time-series dashboards
- **Create a location-ordered projection** for spatial/map dashboards
- **Create a lat/long-ordered projection** on `dim_location` for spatial joins
- **Always run `ANALYZE_STATISTICS`** after loading data

### Cost vs Benefit

- **Cost:** Each projection uses disk space (roughly 1x per projection)
- **Benefit:** Queries can be 50-100x faster for the right access pattern

### How Vertica Picks Projections

You don't need to specify which projection to use in your queries. Vertica's optimizer automatically selects the best projection based on:
- `WHERE` clause predicates
- `JOIN` keys
- `GROUP BY` columns
- `ORDER BY` columns
- Statistics on data distribution

### Important Commands

After creating projections:
```sql
-- Populate the new physical sort orders
SELECT REFRESH('dim_date, dim_location, dim_disease, fact_infection');

-- Give the optimizer cardinality estimates
SELECT ANALYZE_STATISTICS('dim_date');
SELECT ANALYZE_STATISTICS('dim_location');
SELECT ANALYZE_STATISTICS('dim_disease');
SELECT ANALYZE_STATISTICS('fact_infection');
```

**Note:** These commands have already been run during initialization.

---

## Exercise: Find All Infections in Chiang Mai

Write a query that finds all infections in Chiang Mai province during 2024. Use `explain_query()` to check the execution plan.

**Hints:**
- Filter by `province_name_th = 'เชียงใหม่'` (Chiang Mai in Thai)
- Filter by `year_num = 2024`
- Join `fact_infection`, `dim_location`, `dim_disease`, and `dim_date`
- Group by district and disease
- Check which projections Vertica uses

In [ ]:
# Your code here:
# Step 1: Check the execution plan
explain_query("""
-- Write your query here
""")

In [ ]:
# Step 2: Run the actual query
run_query("""
-- Write your query here
""")

<details>
<summary><b>Click to see the solution</b></summary>

```python
# Step 1: Check the execution plan
explain_query("""
SELECT
    loc.district_name_th,
    dis.disease_name_th,
    COUNT(*)                          AS case_count,
    AVG(fi.severity_score)            AS avg_severity,
    SUM(CASE WHEN fi.outcome = 'deceased' THEN 1 ELSE 0 END) AS deaths
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
WHERE loc.is_current = TRUE
  AND loc.province_name_th = 'เชียงใหม่'
  AND dd.year_num = 2024
GROUP BY loc.district_name_th, dis.disease_name_th
ORDER BY case_count DESC
""")

# Step 2: Run the actual query
run_query("""
SELECT
    loc.district_name_th,
    dis.disease_name_th,
    COUNT(*)                          AS case_count,
    AVG(fi.severity_score)            AS avg_severity,
    SUM(CASE WHEN fi.outcome = 'deceased' THEN 1 ELSE 0 END) AS deaths
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
WHERE loc.is_current = TRUE
  AND loc.province_name_th = 'เชียงใหม่'
  AND dd.year_num = 2024
GROUP BY loc.district_name_th, dis.disease_name_th
ORDER BY case_count DESC
LIMIT 20
""")
```

**What to observe in the EXPLAIN output:**
- Vertica should use `dim_location_by_sk` or `dim_location_spatial` (both work well for province filters)
- Vertica should use `fact_infection_by_location` (sorted by location_sk) or `fact_infection_by_date` (sorted by date)
- You should see MERGE JOIN operations
- The province filter should reduce rows early in the plan

</details>

---

## Summary

### What We Learned

1. **Projections are sorted physical copies** of your data
2. You can have **MULTIPLE projections per table** (no extra query syntax needed)
3. Vertica's optimizer **picks the best projection automatically**
4. For spatial queries: **ORDER BY latitude, longitude** creates data locality
5. This replaces the need for spatial indexes (R-tree)

### Key Takeaway

**Data locality through sort order** is Vertica's answer to spatial indexing. By physically storing geographically close records together on disk, Vertica achieves similar performance to spatial indexes without the overhead at query time.

### Next Steps

In Module 3, we'll explore how SCD Type 2 handles boundary changes when subdistrict boundaries are redrawn.